In [1]:
import matplotlib.pyplot as plt   # plotting
import numpy as np

from tensorflow.keras.layers import Input, Flatten, Dense, Dropout, BatchNormalization, TimeDistributed, LSTM
from tensorflow.keras.layers import Conv2D, DepthwiseConv2D, Activation, AveragePooling2D, SeparableConv2D
from tensorflow.keras.models import Model
from tensorflow.keras.constraints import max_norm


import sys

import os
os.chdir("/Volumes/Academico/2021-TIC phd/eeg/participants_cnn/")

In [2]:
def modelar_cnn_online(X_train, y_train, plot_curves=True):
    Samples = X_train.shape[2]  # timesteps
    Chans = X_train.shape[1]  # num channels, two electrodes

    nb_classes = 2 # happy, sad
    dropoutRate = 0.5
    kernLength = 50 # 0.5 x sampling rate
    F1 = 8
    D = 4  # Default is D=2
    F2 = 16
    
    # define model
    input1   = Input(shape = (Chans, Samples, 1))

    ##################################################################
    block1       = Conv2D(F1, (1, kernLength), padding = 'same',
                                   input_shape = (Chans, Samples, 1),
                                   use_bias = False)(input1)
    block1       = BatchNormalization()(block1)
    block1       = DepthwiseConv2D((Chans, 1), use_bias = False, 
                                   depth_multiplier = D,
                                   depthwise_constraint = max_norm(1.))(block1)
    block1       = BatchNormalization()(block1)
    block1       = Activation('elu')(block1)
    block1       = AveragePooling2D((1, 4))(block1)
    block1       = Dropout(dropoutRate)(block1)
    
    block2       = SeparableConv2D(F2, (1, 16), use_bias = False, padding = 'same')(block1)
    block2       = BatchNormalization()(block2)
    block2       = Activation('elu')(block2)
    block2       = AveragePooling2D((1, 8))(block2)
    block2       = Dropout(dropoutRate)(block2)
    
    flatten      = TimeDistributed(Flatten(name = 'flatten'))(block2)

    lstm         = LSTM(F2, return_sequences=False)(flatten) 
    
    dense        = Dense(nb_classes, activation='sigmoid')(lstm)
    
    Output       = Dense(1,activation='sigmoid')(dense)

    model= Model(input1, Output)

    # Compile the model
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    #print(model.summary())
    
    # Train the model
    History = model.fit(X_train, y_train, epochs=10, verbose=1)
    
    if plot_curves:
        fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(9,3))
        axs[0].plot(History.history['loss'], color='b', label='Training loss')
        axs[0].set_title("Loss curves")
        axs[0].legend(loc='best', shadow=True)
        axs[1].plot(History.history['accuracy'], color='b', label='Training accuracy')
        axs[1].set_title("Accuracy curves")
        axs[1].legend(loc='best', shadow=True)
        plt.show()
        
    # Evaluate the model
    loss, accuracy = model.evaluate(X_train, y_train)
    print('timesteps:',Samples,', Train Loss:', loss, ', Train Accuracy:', accuracy)
    
    return model

In [ ]:
# participant = input("Write the participant CODE to load the training datasets and press Enter:")
# tomo el código del participante de la linea de comantos
participant = sys.argv[1]

#participant = "p07"
X_train = np.load(participant + '_X_train.npy')
y_train = np.load(participant + '_y_train.npy')

modelo = modelar_cnn_online(X_train, y_train, plot_curves=False)

modelo.save(participant + '_eegnet_model.h5')